# Experimento — Projeto_MLP (Luciano)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpeuzinCosta/kws-borda/blob/main/notebooks/experimentos/Projeto_MLP.ipynb)

MLP em **12 classes** (10 comandos + unknown + silence), MFCC 10×49, normalização por amostra — **79,6% val**.

Notebook autônomo: baixa o GSC v2 sozinho em `/content/data` e não depende da bancada do repositório.

In [ ]:
import random
from pathlib import Path

import torch
import torchaudio
import torch.nn.functional as F

from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchaudio.datasets import SPEECHCOMMANDS
from torchaudio.transforms import MFCC

import os

In [ ]:
# Faz o download do Google Speech Commands v0.02.
# O download ocorre somente na primeira execução.


if not os.path.exists("/content/data/SpeechCommands/speech_commands_v0.02"):
    dataset = SPEECHCOMMANDS(
        root=DATA_ROOT,
        url="speech_commands_v0.02",
        download=True
    )

In [ ]:
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

DATA_ROOT = Path("/content/data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16_000
AUDIO_LENGTH = SAMPLE_RATE  # 1 segundo

COMMANDS = [
    "yes",
    "no",
    "up",
    "down",
    "left",
    "right",
    "on",
    "off",
    "stop",
    "go"
]

CLASSES = COMMANDS + ["unknown", "silence"]

label_to_idx = {
    label: index
    for index, label in enumerate(CLASSES)
}

print(label_to_idx)

{'yes': 0, 'no': 1, 'up': 2, 'down': 3, 'left': 4, 'right': 5, 'on': 6, 'off': 7, 'stop': 8, 'go': 9, 'unknown': 10, 'silence': 11}


In [ ]:
class KWSDataset(Dataset):

    def __init__(self, root, subset):

        dataset = SPEECHCOMMANDS(
            root=root,
            url="speech_commands_v0.02",
            subset=subset,
            download=False
        )

        self.mfcc = MFCC(
            sample_rate=SAMPLE_RATE,
            n_mfcc=10,
            melkwargs={"n_fft": 640, "hop_length": 320, "n_mels": 40, "center": False}
        )
        # saída: (1, 10, 49) -> 490 valores

        rng = random.Random(SEED)

        # ---------- Amostras de classes e pool de unknown ----------
        class_indices = []
        unknown_pool = []

        for i in range(len(dataset)):
            _, _, label, _, _ = dataset.get_metadata(i)

            if label in COMMANDS:
                class_indices.append(i)
            else:
                unknown_pool.append(i)

        samples_per_class = len(class_indices) // len(COMMANDS)

        # ---------- Amostras unknown ----------
        unknown_indices = rng.sample(unknown_pool, min(samples_per_class, len(unknown_pool)))

        # ---------- Monta waveforms e labels ----------
        self.waveforms = []
        self.labels = []

        for i in class_indices:
            waveform, _, label, _, _ = dataset[i]
            self.waveforms.append(waveform)
            self.labels.append(label)

        for i in unknown_indices:
            waveform, _, _, _, _ = dataset[i]
            self.waveforms.append(waveform)
            self.labels.append("unknown")

        # ---------- Amostras silence ----------
        noise_folder = Path(root) / "SpeechCommands" / "speech_commands_v0.02" / "_background_noise_"

        noises = []
        for file in sorted(noise_folder.glob("*.wav")):
            noise, _ = torchaudio.load(file)
            noises.append(noise)

        for _ in range(samples_per_class):
            noise = noises[rng.randrange(len(noises))]
            start = rng.randint(0, noise.shape[1] - AUDIO_LENGTH)
            self.waveforms.append(0.1 * noise[:, start:start + AUDIO_LENGTH])
            self.labels.append("silence")

    def __len__(self):
        return len(self.waveforms)

    def __getitem__(self, i):
        waveform = self.waveforms[i]
        label = self.labels[i]

        waveform = waveform[:, :AUDIO_LENGTH]
        if waveform.shape[1] < AUDIO_LENGTH:
            waveform = F.pad(waveform, (0, AUDIO_LENGTH - waveform.shape[1]))

        features = self.mfcc(waveform)

        mean = features.mean(dim=2, keepdim=True)
        std = features.std(dim=2, keepdim=True)
        features = (features - mean) / (std + 1e-6)

        return features, label_to_idx[label]

In [ ]:
class KWSDataLoaders:

    def __init__(self, root, batch_size=128):

        self.batch_size = batch_size

        self.train_dataset = KWSDataset(
            root=root,
            subset="training"
        )

        self.val_dataset = KWSDataset(
            root=root,
            subset="validation"
        )

        self.test_dataset = KWSDataset(
            root=root,
            subset="testing"
        )

    def train_dataloader(self):

        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True
        )

    def val_dataloader(self):

        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False
        )

    def test_dataloader(self):

        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False
        )

In [ ]:
data = KWSDataLoaders(
    root=DATA_ROOT,
    batch_size=128
)

train_loader = data.train_dataloader()
val_loader = data.val_dataloader()
test_loader = data.test_dataloader()

In [ ]:
features, targets = next(iter(train_loader))

print(features.shape)
print(targets.shape)

torch.Size([128, 1, 10, 49])
torch.Size([128])


In [ ]:
#features

In [ ]:
len(train_loader)

289

In [ ]:
class MLP(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(10 * 49, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 128),     nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 128),     nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, len(CLASSES))
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = MLP().to(device)

loss_function = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

number_epochs = 10

In [ ]:
for epoch in range(number_epochs):

    # Treinamento
    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for batch_index, (features, targets) in enumerate(
        train_loader,
        start=1
    ):

        features = features.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        logits = model(features)

        loss = loss_function(
            logits,
            targets
        )

        loss.backward()
        optimizer.step()

        predictions = torch.argmax(
            logits,
            dim=1
        )

        batch_correct = (
            predictions == targets
        ).sum().item()

        batch_accuracy = (
            batch_correct / targets.size(0)
        )

        train_loss += loss.item() * targets.size(0)
        train_correct += batch_correct
        train_total += targets.size(0)

        if batch_index % 10 == 0:
            print(
                f"Época {epoch + 1:02d} | "
                f"Treino | "
                f"Batch {batch_index}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f} | "
                f"Acurácia: {batch_accuracy:.4f}"
            )

    train_loss = train_loss / train_total
    train_accuracy = train_correct / train_total

    # Validação
    model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for batch_index, (features, targets) in enumerate(
            val_loader,
            start=1
        ):

            features = features.to(device)
            targets = targets.to(device)

            logits = model(features)

            loss = loss_function(
                logits,
                targets
            )

            predictions = torch.argmax(
                logits,
                dim=1
            )

            batch_correct = (
                predictions == targets
            ).sum().item()

            batch_accuracy = (
                batch_correct / targets.size(0)
            )

            val_loss += loss.item() * targets.size(0)
            val_correct += batch_correct
            val_total += targets.size(0)

            if batch_index % 10 == 0:
                print(
                    f"Época {epoch + 1:02d} | "
                    f"Validação | "
                    f"Batch {batch_index}/{len(val_loader)} | "
                    f"Loss: {loss.item():.4f} | "
                    f"Acurácia: {batch_accuracy:.4f}"
                )

    val_loss = val_loss / val_total
    val_accuracy = val_correct / val_total

    print(
        f"\nResumo da época {epoch + 1:02d} | "
        f"Loss treino: {train_loss:.4f} | "
        f"Acurácia treino: {train_accuracy:.4f} | "
        f"Loss validação: {val_loss:.4f} | "
        f"Acurácia validação: {val_accuracy:.4f}\n"
    )

Época 01 | Treino | Batch 10/289 | Loss: 2.4637 | Acurácia: 0.0938
Época 01 | Treino | Batch 20/289 | Loss: 2.3836 | Acurácia: 0.2188
Época 01 | Treino | Batch 30/289 | Loss: 2.1949 | Acurácia: 0.2656
Época 01 | Treino | Batch 40/289 | Loss: 1.9317 | Acurácia: 0.3828
Época 01 | Treino | Batch 50/289 | Loss: 1.7957 | Acurácia: 0.4219
Época 01 | Treino | Batch 60/289 | Loss: 1.5695 | Acurácia: 0.4844
Época 01 | Treino | Batch 70/289 | Loss: 1.5767 | Acurácia: 0.4688
Época 01 | Treino | Batch 80/289 | Loss: 1.3157 | Acurácia: 0.5859
Época 01 | Treino | Batch 90/289 | Loss: 1.5209 | Acurácia: 0.4297
Época 01 | Treino | Batch 100/289 | Loss: 1.3444 | Acurácia: 0.5547
Época 01 | Treino | Batch 110/289 | Loss: 1.3135 | Acurácia: 0.5469
Época 01 | Treino | Batch 120/289 | Loss: 1.4039 | Acurácia: 0.4844
Época 01 | Treino | Batch 130/289 | Loss: 1.4926 | Acurácia: 0.5000
Época 01 | Treino | Batch 140/289 | Loss: 1.3301 | Acurácia: 0.5547
Época 01 | Treino | Batch 150/289 | Loss: 1.3305 | Acurác

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

all_preds, all_targets = [], []
model.eval()
with torch.no_grad():
    for features, targets in val_loader:
        preds = model(features.to(device)).argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_targets.extend(targets.tolist())

print(confusion_matrix(all_targets, all_preds))
print(classification_report(all_targets, all_preds, target_names=CLASSES))

[[361   3   0   1  23   0   0   0   0   0   6   3]
 [  3 321   3  21   9   3   3   1   2  28   7   5]
 [  1   5 278   5   4   4   8  17  10   5   6   7]
 [  3  33   2 299   3   1   8   0   5  17   4   2]
 [ 20   2   3   2 295  13   0   2   2   2   4   7]
 [  2   0   3   0  16 322   5   2   0   2   6   5]
 [  0   1  11   9   1   2 310  12   3   5   5   4]
 [  1   0  47   1  12   0  15 284   2   5   1   5]
 [  1   1  19   4   0   0   0   0 314   1   5   5]
 [  1  71  10  34   1   2   4   3   1 232  10   3]
 [ 10  22  14  25  12  21  39   6   9  25 181   6]
 [  2   1   5   4  10   2   1   0   2   0   3 340]]
              precision    recall  f1-score   support

         yes       0.89      0.91      0.90       397
          no       0.70      0.79      0.74       406
          up       0.70      0.79      0.75       350
        down       0.74      0.79      0.76       377
        left       0.76      0.84      0.80       352
       right       0.87      0.89      0.88       363
        